# Train Modified Nestnet full final head on Kaggle - 4-fold CV + fixed-test evaluation

Trains the **Modified Nestnet (UNet++ with YOLOX bounding-box priors)** using the full final UNet++ head. This notebook overrides the current `mod_nestnet` preset that trains the near-last head (`--ds-train-output-index 2`) and instead runs `--ds-train-head last --ds-inference last`.

Outputs are isolated under `runs/cv_mod_nestnet_full`, so this experiment does not overwrite or get skipped by the existing Mod NestNet deep-supervision/near-last-head runs under `runs/cv`.

## Before you run
1. **Accelerator:** Settings -> **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings -> **On** (for `git`, `pip`).
3. **Data:** right panel -> *Add Input* -> your **pbl4-splits** dataset. This single dataset must include both `splits/` and `bb_maps/yolox/` (rebuild with `cd data && zip -r -0 splits.zip splits bb_maps -x '*/yolox_coco/*'` if your current one only has `splits/`).

Then **Run All**, top to bottom. Re-running is safe: each cell re-establishes its own state, and training skips folds that have completed with `last.keras`. Interrupted folds can resume from the training backup created by the process-restart loop.

The run still uses mixed precision, the 5-epoch GPU-pool restart loop, and early stopping (patience 4) from `project_presets.py`; only the Mod NestNet head selection is overridden.

## To download results afterwards
Files in `/kaggle/working` only appear in the **Output tab** after you save a version. Use **Save Version -> Quick Save** (with *Save output* on). Do **not** use *Save & Run All* just to download because it retrains.


## 1. Configure

In [ ]:
REPO_URL = "https://github.com/Huay0804/PBL4.git"  # private? use https://<TOKEN>@github.com/Huay0804/PBL4.git
REPO_DIR = "/kaggle/working/PBL4"
MODEL    = "mod_nestnet"   # wired for this notebook; don't change
CV_ROOT  = "runs/cv_mod_nestnet_full"
ARTIFACT_PREFIX = "mod_nestnet_full"


## 2. Get the latest code and use Kaggle's preinstalled stack
Clones if missing, then **always fast-forwards to `origin/main`** so you never
run stale scripts. `git reset --hard` only touches tracked files — your `runs/`
and `data/` outputs are left alone. Uses Kaggle's preinstalled TensorFlow +
Keras 3 (we deliberately do **not** pip-upgrade numpy/TF: scipy and
scikit-image on Kaggle are built against the preinstalled numpy ABI, so an
upgrade leaves them half-broken and unrecoverable in-session).

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("code:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

try:
    import tensorflow as tf
    import keras
except Exception as e:
    raise SystemExit(
        f"Environment broken (TF import failed): {e}\n\n"
        "Stop session (top-right) and start a fresh one — a previous run may "
        "have pip-upgraded numpy/TF and left a half-broken install."
    )
print(f"TF {tf.__version__} | Keras {keras.__version__}")
print("GPUs:", tf.config.list_physical_devices("GPU") or "NONE — Settings → Accelerator")
assert keras.__version__.startswith("3."), f"Need Keras 3.x; have {keras.__version__}"

## 3. Wire up data and sanity-check the environment
`ensure_splits()` symlinks `data/splits` → the mounted splits dataset (auto-
detected under `/kaggle/input` by globbing for `class_map.txt`). `ensure_bb_maps()` does the same for the YOLOX BB priors (hardcoded to look inside the pbl4-splits dataset at `<dataset>/bb_maps/yolox/`). Helpers
are redefined in every data-touching cell so any cell can be run on its own
after a kernel restart.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input - "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)


def ensure_bb_maps():
    """Idempotently link data/bb_maps/yolox -> the BB-priors tree shipped
    INSIDE the pbl4-splits dataset (bb_maps/ is a sibling of splits/ in the
    uploaded zip). Hardcoded path, no separate Kaggle dataset needed."""
    os.chdir(REPO_DIR)
    target = "data/bb_maps/yolox"
    if os.path.exists(os.path.join(target, "folds", "fold_0", "train", "bb_maps")):
        return
    splits_dir = _splits_root()                       # /kaggle/input/<slug>/splits
    dataset_root = os.path.dirname(splits_dir)        # /kaggle/input/<slug>
    bb_root = os.path.join(dataset_root, "bb_maps", "yolox")
    sentinel = os.path.join(bb_root, "folds", "fold_0", "train", "bb_maps")
    if not os.path.exists(sentinel):
        raise FileNotFoundError(
            f"yolox BB priors missing at {bb_root}\n"
            "Rebuild your pbl4-splits dataset to include data/bb_maps/yolox "
            "alongside data/splits - e.g. locally run:\n"
            "    cd data && zip -r -0 splits.zip splits bb_maps -x '*/yolox_coco/*'\n"
            "then re-upload as the pbl4-splits dataset."
        )
    os.makedirs("data/bb_maps", exist_ok=True)
    link = target
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(bb_root, link)
    print("linked data/bb_maps/yolox ->", bb_root)

ensure_splits()
ensure_bb_maps()

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import tensorflow as tf, keras
print(f"TF {tf.__version__} | Keras {keras.__version__}")
print("GPUs:", tf.config.list_physical_devices("GPU") or "NONE - confirm Settings -> Accelerator")

from segmentation_models import ModifiedNestnet
_m = ModifiedNestnet(
    input_shape=(512, 1024, 3),
    classes=33,
    activation="softmax",
    bb_channels=33,
    deep_supervision=False,
    deep_supervision_output_index=3,
)
print(f"mod_nestnet full-head params: {_m.count_params():,}")
print("outputs:", _m.output_names)
del _m


## 4. Train all 4 CV folds
This notebook explicitly overrides the current Mod NestNet preset:

- `--ds-train-head last`: build and train only the final full UNet++ output head.
- `--ds-inference last`: evaluate that same final output.
- no `--ds-train-output-index`: do not train the near-last shortcut head.

The experiment writes to `runs/cv_mod_nestnet_full`. Folds are skipped only when a completed `last.keras` exists, not when an early `best.keras` exists, so interrupted folds can continue correctly.


In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input - "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)


def ensure_bb_maps():
    """Idempotently link data/bb_maps/yolox -> the BB-priors tree shipped
    INSIDE the pbl4-splits dataset (bb_maps/ is a sibling of splits/ in the
    uploaded zip). Hardcoded path, no separate Kaggle dataset needed."""
    os.chdir(REPO_DIR)
    target = "data/bb_maps/yolox"
    if os.path.exists(os.path.join(target, "folds", "fold_0", "train", "bb_maps")):
        return
    splits_dir = _splits_root()                       # /kaggle/input/<slug>/splits
    dataset_root = os.path.dirname(splits_dir)        # /kaggle/input/<slug>
    bb_root = os.path.join(dataset_root, "bb_maps", "yolox")
    sentinel = os.path.join(bb_root, "folds", "fold_0", "train", "bb_maps")
    if not os.path.exists(sentinel):
        raise FileNotFoundError(
            f"yolox BB priors missing at {bb_root}\n"
            "Rebuild your pbl4-splits dataset to include data/bb_maps/yolox "
            "alongside data/splits - e.g. locally run:\n"
            "    cd data && zip -r -0 splits.zip splits bb_maps -x '*/yolox_coco/*'\n"
            "then re-upload as the pbl4-splits dataset."
        )
    os.makedirs("data/bb_maps", exist_ok=True)
    link = target
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(bb_root, link)
    print("linked data/bb_maps/yolox ->", bb_root)

def fold_complete(fold_root):
    if not os.path.isdir(fold_root):
        return False
    for _, _, files in os.walk(fold_root):
        if "last.keras" in files:
            return True
    return False

ensure_splits()
ensure_bb_maps()

os.environ["PBL4_GPU_DISPLAY_RESERVE_MB"] = "0"
os.environ["PBL4_CV_OUTPUT_DIR"] = CV_ROOT

for k in range(4):
    fold_root = f"{CV_ROOT}/fold_{k}/mod_nestnet"
    if fold_complete(fold_root):
        print(f"skip mod_nestnet full-head fold {k} (last.keras already present under {fold_root})")
        continue
    print(f"\n========== TRAIN mod_nestnet full-head fold {k} ==========", flush=True)
    cmd = (
        "python -u scripts/train_segmentation_cv.py "
        f"--model mod_nestnet --fold {k} "
        "--ds-train-head last --ds-inference last"
    )
    rc = os.system(cmd)
    if rc != 0:
        raise SystemExit(f"mod_nestnet full-head fold {k} training failed (exit {rc}).")
print("\nAll requested folds complete or resumable checkpoints updated.")


## 5. Evaluate each fold on the fixed test set
Evaluation uses the existing `evaluate_final.py` format and points `--run-dir` at the isolated full-head CV root for each fold.


In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input - "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)


def ensure_bb_maps():
    """Idempotently link data/bb_maps/yolox -> the BB-priors tree shipped
    INSIDE the pbl4-splits dataset (bb_maps/ is a sibling of splits/ in the
    uploaded zip). Hardcoded path, no separate Kaggle dataset needed."""
    os.chdir(REPO_DIR)
    target = "data/bb_maps/yolox"
    if os.path.exists(os.path.join(target, "folds", "fold_0", "train", "bb_maps")):
        return
    splits_dir = _splits_root()                       # /kaggle/input/<slug>/splits
    dataset_root = os.path.dirname(splits_dir)        # /kaggle/input/<slug>
    bb_root = os.path.join(dataset_root, "bb_maps", "yolox")
    sentinel = os.path.join(bb_root, "folds", "fold_0", "train", "bb_maps")
    if not os.path.exists(sentinel):
        raise FileNotFoundError(
            f"yolox BB priors missing at {bb_root}\n"
            "Rebuild your pbl4-splits dataset to include data/bb_maps/yolox "
            "alongside data/splits - e.g. locally run:\n"
            "    cd data && zip -r -0 splits.zip splits bb_maps -x '*/yolox_coco/*'\n"
            "then re-upload as the pbl4-splits dataset."
        )
    os.makedirs("data/bb_maps", exist_ok=True)
    link = target
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(bb_root, link)
    print("linked data/bb_maps/yolox ->", bb_root)

ensure_splits()
ensure_bb_maps()

for k in range(4):
    fold_root = f"{CV_ROOT}/fold_{k}/mod_nestnet"
    print(f"\n========== EVAL mod_nestnet full-head fold {k} ==========", flush=True)
    rc = os.system(f"python -u scripts/evaluate_final.py --model mod_nestnet --run-dir {fold_root}")
    if rc != 0:
        raise SystemExit(f"mod_nestnet full-head fold {k} evaluation failed (exit {rc}).")
print("\nAll evaluations done.")


## 6. Results
Cross-validation aggregate (validation folds) from the CV summary, plus the
per-fold test-set summaries.

In [ ]:
import glob, json, os
os.chdir(REPO_DIR)

cv = f"{CV_ROOT}/mod_nestnet_cv_summary.json"
if os.path.exists(cv):
    agg = (json.load(open(cv)) or {}).get("aggregate", {})
    print("=== CV aggregate (validation) ===")
    print(json.dumps(agg, indent=2))
else:
    print(f"(no {cv} yet)")

print("\n=== Per-fold test-set summaries ===")
for p in sorted(glob.glob(f"{CV_ROOT}/fold_*/mod_nestnet/**/test_summary.json", recursive=True)):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2))


## 7. Package results for download
Writes **two** zips to `/kaggle/working`:
- `mod_nestnet_full_models.zip` - only the `best.keras` checkpoints.
- `mod_nestnet_full_results.zip` - metrics, CV summary, and metadata from `runs/cv_mod_nestnet_full` with heavy checkpoints/log backups excluded.

Use Kaggle's right panel -> Output -> `/kaggle/working` to download the zips after a Quick Save.


In [ ]:
import os, glob, subprocess
os.chdir(REPO_DIR)

models_zip  = f"/kaggle/working/{ARTIFACT_PREFIX}_models.zip"
results_zip = f"/kaggle/working/{ARTIFACT_PREFIX}_results.zip"
for z in (models_zip, results_zip):
    if os.path.exists(z):
        os.remove(z)

best_files = sorted({
    p for p in glob.glob(f"{CV_ROOT}/fold_*/mod_nestnet/**/best.keras", recursive=True)
    if "/.training_backup/" not in p
})
if not best_files:
    raise SystemExit(f"No best.keras under {CV_ROOT}/fold_*/mod_nestnet/ - run training and eval first.")

subprocess.run(["zip", "-q", models_zip, *best_files], check=True)
subprocess.run([
    "zip", "-r", "-q", results_zip, CV_ROOT,
    "-x", "*.keras", "*/logs/*", "*/.training_backup/*",
], check=True)

print(f"models  : {models_zip}  ({os.path.getsize(models_zip)/1e6:.1f} MB, {len(best_files)} best.keras)")
print(f"results : {results_zip} ({os.path.getsize(results_zip)/1e6:.1f} MB)")
print("\nDownload:")
print("  - right panel -> Output -> /kaggle/working -> download icon next to each zip")
print("  - or Save Version -> Quick Save (Save output) -> version Output tab -> Download")
